# 🚗 Car Detection YOLO - GPU Training
Run this notebook in Google Colab with GPU for fast training.
**Runtime → Change runtime type → GPU**

In [ ]:
# Mount Google Drive (optional, to save models persistently)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q ultralytics pycocotools albumentations
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
import yaml
import shutil
import json
import zipfile
import urllib.request
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('/content/data')
DATASET_DIR = DATA_DIR / 'car_dataset'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Vehicle classes from COCO
CLASS_MAP = {3: 'car', 4: 'motorcycle', 6: 'bus', 8: 'truck'}
CLASS_NAMES = ['car', 'motorcycle', 'bus', 'truck']
CLASS_ID_MAP = {3: 0, 4: 1, 6: 2, 8: 3}

# Download COCO train2017 (18GB - large but comprehensive)
# OR use val2017 for faster setup (~1GB)
USE_TRAIN = False  # Set to True for full training set

if USE_TRAIN:
    urllib.request.urlretrieve(
        'http://images.cocodataset.org/zips/train2017.zip',
        DATA_DIR / 'train2017.zip'
    )
else:
    urllib.request.urlretrieve(
        'http://images.cocodataset.org/zips/val2017.zip',
        DATA_DIR / 'val2017.zip'
    )

urllib.request.urlretrieve(
    'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
    DATA_DIR / 'annotations_trainval2017.zip'
)

In [ ]:
# Extract
if USE_TRAIN:
    with zipfile.ZipFile(DATA_DIR / 'train2017.zip', 'r') as z:
        z.extractall(DATA_DIR)
    img_dir = DATA_DIR / 'train2017'
    split_name = 'train2017'
else:
    with zipfile.ZipFile(DATA_DIR / 'val2017.zip', 'r') as z:
        z.extractall(DATA_DIR)
    img_dir = DATA_DIR / 'val2017'
    split_name = 'val2017'

ann_dir = DATA_DIR / 'annotations'
if not ann_dir.exists():
    with zipfile.ZipFile(DATA_DIR / 'annotations_trainval2017.zip', 'r') as z:
        z.extractall(DATA_DIR)

In [ ]:
# Filter COCO for vehicle classes only
ann_file = ann_dir / f'instances_{split_name}.json'
with open(ann_file) as f:
    coco = json.load(f)

vehicle_cat_ids = list(CLASS_MAP.keys())
img_id_to_info = {img['id']: img for img in coco['images']}

img_vehicle_anns = {}
for ann in tqdm(coco['annotations'], desc='Filtering annotations'):
    if ann['category_id'] in vehicle_cat_ids and not ann.get('iscrowd', 0):
        img_id = ann['image_id']
        if img_id not in img_vehicle_anns:
            img_vehicle_anns[img_id] = []
        img_vehicle_anns[img_id].append(ann)

print(f'Found {len(img_vehicle_anns)} images with vehicles')

In [ ]:
# Create YOLO dataset
for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

img_ids = list(img_vehicle_anns.keys())
split_idx = int(len(img_ids) * 0.9)  # 90-10 split

for i, img_id in enumerate(tqdm(img_ids, desc='Preparing dataset')):
    img_info = img_id_to_info[img_id]
    src_img = img_dir / img_info['file_name']
    if not src_img.exists():
        continue
    
    split = 'train' if i < split_idx else 'val'
    shutil.copy(src_img, DATASET_DIR / 'images' / split / img_info['file_name'])
    
    lines = []
    for ann in img_vehicle_anns[img_id]:
        cls_id = CLASS_ID_MAP[ann['category_id']]
        x, y, w, h = ann['bbox']
        img_w, img_h = img_info['width'], img_info['height']
        lines.append(f"{cls_id} {(x+w/2)/img_w:.6f} {(y+h/2)/img_h:.6f} {w/img_w:.6f} {h/img_h:.6f}")
    
    with open(DATASET_DIR / 'labels' / split / (Path(img_info['file_name']).stem + '.txt'), 'w') as f:
        f.write('\n'.join(lines))

yaml_content = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
with open(DATASET_DIR / 'dataset.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

t = len(list((DATASET_DIR / 'images' / 'train').iterdir()))
v = len(list((DATASET_DIR / 'images' / 'val').iterdir()))
print(f'Dataset: {t} train, {v} val images')

In [ ]:
# Train with optimal hyperparameters
from ultralytics import YOLO

model = YOLO('yolo11m.pt')  # Start with medium model

results = model.train(
    data=str(DATASET_DIR / 'dataset.yaml'),
    epochs=300,
    patience=30,
    batch=32,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    close_mosaic=10,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,
    device=0,
    project='car_detection',
    name='yolo11m_car',
    exist_ok=True,
    pretrained=True,
    plots=True,
    val=True,
    save=True,
    workers=4,
    seed=42,
)

In [ ]:
# Save best model to Drive
import shutil
best = Path('car_detection') / 'yolo11m_car' / 'weights' / 'best.pt'
if best.exists():
    shutil.copy(best, '/content/drive/MyDrive/car_detection_best.pt')
    print(f'Model saved! Best mAP50: {results.box.map50:.4f}')
    # Download directly
    from google.colab import files
    files.download(best)

In [ ]:
# Validate the model
metrics = model.val(data=str(DATASET_DIR / 'dataset.yaml'), device=0)
print('\nFinal Results:')
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.p:.4f}')
print(f'  Recall:    {metrics.box.r:.4f}')

for i, name in enumerate(CLASS_NAMES):
    ap = metrics.box.ap[i]
    print(f'  {name}: mAP50={ap[0]:.4f}')